# Optimizing Model Parameters

In [36]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [37]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [38]:
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
)

In [39]:
train_dataloader = DataLoader(training_data, batch_size = 64)
test_dataloader = DataLoader(test_data, batch_size = 64)

### Define the Network

In [40]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()     ##image is 28*28(2-D), for linear layer convert to 784(1-D)
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

### Hyperparameters
- Input size
- No. of iterations(epochs)
- Learning rate(alpha)

In [41]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

### The Loop
- Training loop: try to get desired outputs
- Testing loop: evluate model's performance

Loss function: measures dissimilarity b.w. obtained and target value

In [42]:
loss_fn = nn.CrossEntropyLoss()

Optimizer: adjust parameters to reduce model error

In [43]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

### Train and test loops

In [46]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()       ##added for best practices
    for batch, (X, y) in enumerate(dataloader):     ##enumerate gives both batch number and x(64 images in a batch) and y(correct labels)
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()             ##backpropagate prediction loss
        optimizer.step()            ##adjust parameters to reduce loss
        optimizer.zero_grad()       ##reset gradients at each iteration to prevent double-counting(PyTorch accumulates gradients by default)

        if batch%100 == 0:          ##print after every 100 batches
            loss, current = loss.item(), batch*batch_size + len(X)
            print(f"Loss: {loss:>7f}, [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():           ##skip gradient computations during testing(no backpropagation during testing)
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()    ##accumulate loss incurred in every batch

            '''THE FOLLOWING CODE LINE IS CONFUSING, BREAKDOWN:
            - get the highest score(pred.argmax(1))
            - check if pred matches with actual values, output is in true/false(== y)
            - convert true/false to numbers(.type(troch.float))
            - count correct predictions(.sum), convert from tensor to python number(.item)
            '''
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches        ##calculate average test loss
    correct /= size                 ##calculate accuracy
    print(f"Test error: \n Acuuracy: {100*correct:>0.1f}, Avg loss: {test_loss:>8f} \n")

In [45]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done")

Epoch 1
-----------------------------
Loss: 2.302289, [   64/60000]
Loss: 2.300396, [ 6464/60000]
Loss: 2.281116, [12864/60000]
Loss: 2.276313, [19264/60000]
Loss: 2.259598, [25664/60000]
Loss: 2.229437, [32064/60000]
Loss: 2.227424, [38464/60000]
Loss: 2.201619, [44864/60000]
Loss: 2.196097, [51264/60000]
Loss: 2.163646, [57664/60000]
Test error: 
 Acuuracy: 50.7, Avg loss: 2.162400 

Epoch 2
-----------------------------
Loss: 2.164231, [   64/60000]
Loss: 2.160461, [ 6464/60000]
Loss: 2.107432, [12864/60000]
Loss: 2.124694, [19264/60000]
Loss: 2.067822, [25664/60000]
Loss: 2.009066, [32064/60000]
Loss: 2.036299, [38464/60000]
Loss: 1.960750, [44864/60000]
Loss: 1.960126, [51264/60000]
Loss: 1.893350, [57664/60000]
Test error: 
 Acuuracy: 55.2, Avg loss: 1.889632 

Epoch 3
-----------------------------
Loss: 1.912228, [   64/60000]
Loss: 1.886346, [ 6464/60000]
Loss: 1.775201, [12864/60000]
Loss: 1.821130, [19264/60000]
Loss: 1.704352, [25664/60000]
Loss: 1.646317, [32064/60000]
Loss